# CMT Agreement

## Import Packages

In [1]:
import pandas as pd
import re

## Functions for reading, cleaning, and calculating

In [2]:
def return_anno(annotation,annotator):
    '''
    given an annotation result and an annotator number (int), returns the annotation created by that annotator
    '''
    for n in [0,1]:
        if annotation[n]['completed_by'] == annotator:
            return annotation[n]

In [3]:
def json_to_df(path):
    '''
    takes path to json file of annotations and returns a df 
    '''
    # load in CMT annotation json and save it to a pandas dataframe
    met_df = pd.read_json(path)
    
    # extracting labels from the annotations column
    met_df['labels'] = met_df.apply(lambda row: row.annotations[0]['result'], axis=1)
    
    # cleaning up file names 
    met_df['filename'] = met_df.apply(lambda row: re.sub(r"^[^_]*-", '', row.file_upload), axis=1)
    met_df['filename'] = met_df.apply(lambda row: re.sub(r"_fixed", '', row.filename[:-4].lower()), axis=1)
    met_df['filename'] = met_df.apply(lambda row: re.sub(r"aboriginal", 'indigenous', row.filename), axis=1)
    
    # creating columns for each annotator
    anno_A = met_df.annotations[0][0]['completed_by'] #finds annotator number associated with Annotator A
    anno_B = met_df.annotations[0][1]['completed_by'] #finds annotator number associated with Annotator B
    met_df['anno_A'] = met_df.apply(lambda row: return_anno(row.annotations,anno_A)['result'], axis=1)
    met_df['anno_B'] = met_df.apply(lambda row: return_anno(row.annotations,anno_B)['result'], axis=1)

    return met_df[['anno_A', 'anno_B']]

In [4]:
def pooled_chance_agreement(df):
    '''
    takes a dataframe with columns 'anno_A' and 'anno_B' and return the pooled probability of chance agreement 
    '''
    # lists of the domains used by each annotator 
    anno_A_domains = []
    for row in df['anno_A']:
        for metaphor in row:
            for domain in metaphor['value']['taxonomy']:
                anno_A_domains.append(domain[0])
    anno_B_domains = []
    for row in df['anno_B']:
        for metaphor in row:
            for domain in metaphor['value']['taxonomy']:
                anno_B_domains.append(domain[0])
    
    # total number of domains used by each annotator 
    anno_A_total = len(anno_A_domains)
    anno_B_total = len(anno_B_domains)
    
    # a list of all unique domains
    all_domains = set(anno_A_domains + anno_B_domains)
    
    # dictionaries of domains, with counts set to zero
    anno_A_domains_dict = dict.fromkeys(all_domains,0)
    anno_B_domains_dict = dict.fromkeys(all_domains,0)
    
    # adding to counts for each domain
    for domain in anno_A_domains:
        anno_A_domains_dict[domain] +=1
    for domain in anno_B_domains:
        anno_B_domains_dict[domain] +=1
    
    # setting sum of probability of chance agreement to zero
    sum_p_e = 0
    for domain in all_domains:
        # probability of chance agreement for that domain
        p_e_d = (anno_A_domains_dict[domain]/anno_A_total)*(anno_B_domains_dict[domain]/anno_B_total)
        # adding to sum
        sum_p_e += p_e_d
    # pooling
    pooled_p_e = sum_p_e/len(all_domains)
    return pooled_p_e

## Interannotator Agreement

In [5]:
# set file paths (THIS IS THE ONLY THING THAT NEEDS TO CHANGE)
file_path_manual_interanno = 'ManualTest.json'
file_path_annotator_interanno = 'AnnotatorTest.json'

# from path to dataframe
manual_interanno_df = json_to_df(file_path_manual_interanno)
annotator_interanno_df = json_to_df(file_path_annotator_interanno)

In [6]:
# testing manual
p_o = 0.44 # from calculated strict agreement 
pooled_p_e = pooled_chance_agreement(manual_interanno_df)
print('pooled chance agreement:', pooled_p_e)
print('cohen\'s kappa for agreement testing on manual:',(p_o-pooled_p_e)/(1-pooled_p_e))

pooled chance agreement: 0.0016152516152516153
cohen's kappa for agreement testing on manual: 0.4390939956705025


In [7]:
# testing primary annotators
p_o = 0.223684211 # from calculated strict agreement 
pooled_p_e = pooled_chance_agreement(annotator_interanno_df)
print('pooled chance agreement:', pooled_p_e)
print('cohen\'s kappa for agreement testing between annotators:',(p_o-pooled_p_e)/(1-pooled_p_e))

pooled chance agreement: 0.00020494635648273912
cohen's kappa for agreement testing between annotators: 0.22352507529328114
